In [4]:
!pwd

/home/jovyan/notebooks/LAB


In [2]:
!kafka-topics.sh --create --topic transactions --bootstrap-server broker:9092 --if-not-exists

In [13]:
%%file producer.py
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction():
    # TWÓJ KOD
    # Zwróć słownik z polami: tx_id, user_id, amount, store, category, timestamp
    
    # {tx_id:"TX0001", user_id:random z u01-u20, amount:random float z (5.0 - 5000.0),
    #  store:random z (“Warszawa”, “Kraków”, “Gdańsk”, “Wrocław”),
    #  category:random z (“elektronika”, “odzież”, “żywność”, “książki”), timestamp:czas ISO}
    
    tx_id = f'TX{random.randint(1000,9999)}'
    user_id = f'u{random.randint(1, 20):02d}'
    amount = round(random.uniform(5.0, 5000.0), 2)
    store = random.choice(('Warszawa', 'Kraków', 'Gdańsk', 'Wrocław'))
    category = random.choice(('elektronika', 'odzież', 'żywność', 'książki'))
    timestamp = datetime.now().isoformat()
    
    return {'tx_id':tx_id, 'user_id':user_id, 'amount':amount,
            'store':store, 'category':category, 'timestamp':timestamp}

# TWÓJ KOD
# Pętla: generuj transakcję, wyślij do tematu 'transactions', wypisz, sleep 1s

for i in range(1000):
    tx = generate_transaction()
    producer.send('transactions', value = tx)
    print(f"[{i+1}] {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']}")
    time.sleep(0.5)

producer.flush()
producer.close()

Overwriting producer.py


In [16]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    tx = message.value
    if tx['amount'] > 3000:
        print(f'ALERT: {tx["tx_id"]} | {tx["amount"]} PLN | {tx["store"]} | {tx["category"]}')
    
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź czy amount > 3000, jeśli tak — wypisz ALERT
# Format: ALERT: TX0042 | 2345.67 PLN | Warszawa | elektronika

# ConsumerRecord(topic='transactions', partition=0, offset=281, 
# timestamp=1774961451173, timestamp_type=0, key=None, 
# value={'tx_id': 'TX1262', 'user_id': 'u16', 'amount': 4530.77, 'store': 'Wrocław', 
# 'category': 'elektronika', 'timestamp': '2026-03-31T12:50:51.173300'}, headers=[], 
# checksum=None, serialized_key_size=-1, serialized_value_size=151, serialized_header_size=-1)


Overwriting consumer_filter.py


In [15]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id = 'risk_assessment',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Dodaje poziom ryzyka transakcji")

for message in consumer:
    tx = message.value
    if tx['amount'] > 3000:
        risk_level = 'HIGH'
    elif tx['amount'] > 1000:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'LOW'
        
    tx['risk_level'] = risk_level
    print(tx)

# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
# Dodaj pole risk_level na podstawie amount
# Wypisz wzbogaconą transakcję

Overwriting consumer_enrich.py


In [14]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='analytics',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    tx = message.value
    store = tx['store']
    amount = tx['amount']
    store_counts[store] += 1
    msg_count += 1

    if store not in total_amount:
        total_amount[store] = amount
    else:
        total_amount[store] += amount

    if msg_count == 10:
        print('RAPORT:')
        for city in sorted(total_amount.keys()):
            print(f'{city} | {store_counts[city]} | {total_amount[city]:.2f} PLN | {total_amount[city]/store_counts[city]:.2f} PLN')
        print()
        msg_count = 0
        #store_counts.clear()
        #total_amount.clear()

# TWÓJ KOD
# Dla każdej wiadomości:
#   1. Zwiększ store_counts[store]
#   2. Dodaj amount do total_amount[store]
#   3. Co 10 wiadomości wypisz tabelę:
#      Sklep | Liczba | Suma | Średnia

Overwriting consumer_count.py


In [13]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='analytics',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

category_counts = defaultdict(int)
total_amount = defaultdict(float)
max_amount = defaultdict(float)
min_amount = defaultdict(float)
msg_count = 0

for message in consumer:
    tx = message.value
    category = tx['category']
    amount = tx['amount']
    category_counts[category] += 1
    msg_count += 1
    total_amount[category] += amount
    
    # if min_amount[category] == 0:
    #     min_amount[category] = amount
    
    if amount > max_amount[category]:
        max_amount[category] = amount
        
    if category not in min_amount or amount < min_amount[category]:
        min_amount[category] = amount

    if msg_count == 10:
        print('RAPORT:')
        for cat in sorted(total_amount.keys()):
            print(f'{cat} | {category_counts[cat]} | {total_amount[cat]:.2f} PLN | MIN: {min_amount[cat]:.2f} PLN | MAX: {max_amount[cat]:.2f} PLN')
        print()
        msg_count = 0

# TWÓJ KOD
# śledzi per kategoria: - liczbę transakcji - łączny przychód - min i max kwotę
# co 10 transakcji

Overwriting consumer_stats.py


In [14]:
%%file consumer_anomaly.py
from kafka import KafkaConsumer
import json
from collections import defaultdict
from datetime import datetime, timedelta

# Napisz konsumenta wykrywającego anomalie prędkości: 
# alert jeśli ten sam user_id wykona więcej niż 3 transakcje w ciągu 60 sekund

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

id_counts = defaultdict(list)

for message in consumer:
    # value={'tx_id': 'TX1262', 'user_id': 'u16', 'amount': 4530.77, 'store': 'Wrocław', 
    # 'category': 'elektronika', 'timestamp': '2026-03-31T12:50:51.173300'}
    tx = message.value
    user_id = tx['user_id']
    timestamp = tx['timestamp']
    id_counts[user_id].append(datetime.fromisoformat(timestamp))
    if len(id_counts[user_id]) > 3:
        time_diff = id_counts[user_id][-1] - id_counts[user_id][-4]
        if time_diff < timedelta(seconds=60):
            print(f'ALERT - użytkownik: {user_id} wykonał więcej niż 3 transakcje w ciągu {time_diff.total_seconds():.2f} sek')
            del id_counts[user_id]
        else:
            del id_counts[user_id][0]


Overwriting consumer_anomaly.py
